# Count tokens in your own documents

This notebook counts tokens the way the [Token](https://bradleyboehmke.github.io/ai-engineering-notes/dictionary/token.html) dictionary entry does, with OpenAI's `o200k_base` tokenizer (the GPT-4o family). For each file it also compares the result with the rule of thumb of about 4 characters, or 0.75 words, per token.

**Files it reads**

- Any plain-text file: `.txt`, `.md`, `.py`, `.csv`, `.json`, `.html`, and so on
- Word documents: `.docx` (paragraphs and tables)
- PDFs with a text layer: `.pdf`

**Files it doesn't read:** old Word `.doc` files, spreadsheets such as `.xlsx`, images, and scanned PDFs with no text layer. Save those as `.docx`, `.csv` or text first.

Counts are for `o200k_base` only. Other models use other tokenizers and will give different counts.

## 1. Install the libraries

In [ ]:
%pip install --quiet tiktoken pypdf python-docx

## 2. The code

This is the same code as [`count.py`](https://github.com/bradleyboehmke/ai-engineering-notes/blob/main/learning/llm-mechanics/experiments/count.py). `read_text` turns a file into plain text, and `report` counts its tokens and prints how it compares with the rule of thumb.

In [ ]:
import tiktoken
from docx import Document
from pypdf import PdfReader

enc = tiktoken.get_encoding("o200k_base")

def read_text(path):
    if path.endswith(".pdf"):
        return "\n".join(page.extract_text() for page in PdfReader(path).pages)
    if path.endswith(".docx"):
        doc = Document(path)
        cells = [cell.text for table in doc.tables for row in table.rows for cell in row.cells]
        return "\n".join([p.text for p in doc.paragraphs] + cells)
    return open(path, encoding="utf-8").read()  # any plain-text file

def report(path):
    try:
        text = read_text(path)
    except UnicodeDecodeError:
        print(f"{path}: skipped, not plain text, .docx or .pdf")
        return
    n = len(enc.encode(text))
    if n == 0:
        print(f"{path}: no text found (a scanned PDF has no text layer)")
        return
    print(f"{path}: {n} tokens, {len(text)/n:.1f} chars/token, "
          f"{len(text.split())/n:.2f} words/token")

## 3. Count the sample files

This downloads the three sample files from the term page and counts them. You should get 245, 296 and 325 tokens.

In [ ]:
import urllib.request
from pathlib import Path

BASE = "https://raw.githubusercontent.com/bradleyboehmke/ai-engineering-notes/main/learning/llm-mechanics/experiments/samples/"
SAMPLES = ["quarterly-update.md", "moving_average.py", "supplier-invoice.pdf"]

Path("samples").mkdir(exist_ok=True)
for name in SAMPLES:
    urllib.request.urlretrieve(BASE + name, f"samples/{name}")
    report(f"samples/{name}")

## 4. Count your own files

In Colab, run the cell and pick one or more files to upload.

Outside Colab, put your files next to this notebook and list them in `paths` instead.

In [ ]:
try:
    from google.colab import files
    paths = list(files.upload())
except ImportError:
    paths = []  # outside Colab: e.g. paths = ["my-report.docx"]

for path in paths:
    report(path)

## What to look for

- **chars/token below 4** means the file costs more than the rule of thumb predicts. Numbers, symbols, code and tables usually do.
- **words/token below 0.75** means each word costs more than one token on average.
- For a PDF or Word file, the count depends on how the text was extracted. A different library can give a different count for the same document.